# Module 4: Data Preprocessing

This notebook implements a production-quality preprocessing pipeline for the flood prediction dataset. The objective is to prepare the data for downstream machine learning while preserving modularity, reproducibility, and clarity.

## Step 1: Import Libraries

- `numpy`: Numerical operations and array support.
- `pandas`: Data manipulation and tabular processing.
- `joblib`: Saving and loading preprocessing artifacts.
- `warnings`: Suppressing unnecessary warnings.
- `sklearn`: Machine learning preprocessing tools such as imputation, encoding, scaling, and splitting.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from utils.preprocessing import PreprocessingPipeline

warnings.filterwarnings('ignore')

## Step 2: Load Dataset

The raw Excel file is loaded and inspected to confirm the expected shape and target column.

In [ ]:
project_root = Path.cwd().resolve()
dataset_path = project_root / 'dataset' / 'raw' / 'flood_dataset.xlsx'
pipeline = PreprocessingPipeline(dataset_path, output_dir=project_root)
df = pipeline.load_data()
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Target Column:')
display(df['flood'].head())

## Step 3: Missing Value Detection

The notebook identifies missing values before applying any treatment. Understanding the missingness pattern is important for preserving data quality.

In [ ]:
print(df.isnull())
print(df.isnull().sum())
print((df.isnull().sum() / len(df) * 100).round(2))

## Step 4: Missing Value Handling

Numeric columns are imputed with the median because it is robust to outliers. Categorical columns use the mode because it preserves the most common category.

In [ ]:
cleaned_df = pipeline.clean_data()
cleaned_df = pipeline.handle_missing()
print(cleaned_df.isnull().sum())

## Step 5: Duplicate Handling

Duplicate records are removed to avoid repeated information from affecting downstream analysis and model training.

In [ ]:
before_rows = len(cleaned_df)
cleaned_df = pipeline.remove_duplicates()
after_rows = len(cleaned_df)
print('Rows Before:', before_rows)
print('Rows After:', after_rows)
print('Duplicates Removed:', before_rows - after_rows)

## Step 6: Outlier Detection

The IQR method identifies potential outliers without removing any values. This is useful for understanding extreme observations and preparing for capping.

In [ ]:
numeric_features = [col for col in cleaned_df.columns if col != 'flood' and pd.api.types.is_numeric_dtype(cleaned_df[col])]
for feature in numeric_features:
    lower_bound, upper_bound, outlier_count = pipeline.detect_outliers(feature)
    print(feature, 'lower=', lower_bound, 'upper=', upper_bound, 'outliers=', outlier_count)

## Step 7: Outlier Treatment

IQR capping replaces values beyond acceptable bounds with the upper or lower threshold. This preserves rows while reducing the impact of extreme values.

In [ ]:
cleaned_df = pipeline.cap_outliers()
print(cleaned_df.head())

## Step 8: Categorical Feature Detection and Encoding

Categorical features are detected automatically. Label encoding is applied for classification-friendly transformation, and each encoder is saved for reuse.

In [ ]:
cleaned_df = pipeline.encode()
print(cleaned_df.dtypes)

## Step 9: Feature Selection

The target column is separated from the feature columns so the data can be split into train and test sets.

In [ ]:
X, y = pipeline.split_features_target()

## Step 10: Train-Test Split

The features are divided into training and testing subsets to support later model development.

In [ ]:
X_train, X_test, y_train, y_test = pipeline.train_test_split_data()

## Step 11: Feature Scaling

Standard scaling is applied to the features only, ensuring that each feature contributes proportionally. This is important for many models and helps stabilize training.

In [ ]:
X_train_scaled, X_test_scaled, scaler = pipeline.scale_features()
print('Scaled Training Shape:', X_train_scaled.shape)
print('Scaled Testing Shape:', X_test_scaled.shape)

## Step 12-14: Save Artifacts and Report

The pipeline saves preprocessing artifacts, the processed dataset, and a detailed report for downstream modules.

In [ ]:
pipeline.save()